<a href="https://colab.research.google.com/github/sasvi123/Neural_Network/blob/main/Lab3task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

# 1. Re-load and preprocess the dataset so this cell is independent
housing = fetch_california_housing()
X, y = housing.data, housing.target

# Split data (90% train, 10% test as done in Task 02)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
# (X_test_scaled isn't needed for cross-validation, but good to have)

# 2. Setup the Grid Search
param_grid = {
    'hidden_layer_sizes': [(128, 64), (256, 128, 64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd']
}

grid = GridSearchCV(MLPRegressor(max_iter=300, random_state=42), param_grid, cv=5, n_jobs=-1)

# 3. Fit the grid (This will work now!)
grid.fit(X_train_scaled, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

import pandas as pd
import matplotlib.pyplot as plt

# 1. Convert grid search results into a pandas DataFrame
results = pd.DataFrame(grid.cv_results_)

# 2. Format parameter combinations as a readable string for the y-axis
results['param_combination'] = (
    "Layers: " + results['param_hidden_layer_sizes'].astype(str) +
    " | Act: " + results['param_activation'].astype(str) +
    " | Solver: " + results['param_solver'].astype(str)
)

# 3. Sort by performance so the highest scoring configuration is at the top
results = results.sort_values(by='mean_test_score', ascending=True)

# 4. Generate the horizontal bar chart
plt.figure(figsize=(10, 6))
bars = plt.barh(results['param_combination'], results['mean_test_score'], color='teal', edgecolor='black', alpha=0.8)

# Highlight the best configuration bar in a different color
best_idx = results['mean_test_score'].idxmax()
# Find the position of the best bar after sorting
sorted_best_pos = results.index.get_loc(best_idx)
bars[sorted_best_pos].set_color('darkorange')
bars[sorted_best_pos].set_edgecolor('black')

# 5. Add titles, labels, and baseline indicators
plt.title("Task 04: Hyperparameter Grid Search Performance Comparison", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Mean Cross-Validation $R^2$ Score", fontsize=12)
plt.ylabel("Hyperparameter Configuration", fontsize=12)
plt.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Annotate each bar with its exact score value for easy reading in the report
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, f'{width:.4f}',
             va='center', ha='left', fontsize=10, fontweight='bold' if bar.get_facecolor() == (1.0, 0.549, 0.0, 0.8) else 'normal')

plt.xlim(0, max(results['mean_test_score']) + 0.1) # Add room for data labels
plt.tight_layout()
plt.show()

Best Parameters: {'activation': 'tanh', 'hidden_layer_sizes': (128, 64), 'solver': 'adam'}
Best CV Score: 0.8007034581359063


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
